# Bluestock Fintech — Performance Analytics
This notebook computes daily returns, CAGR, Sharpe, Sortino, Alpha/Beta, Max Drawdown, Fund Scorecard, and Benchmark Comparison for all 40 mutual fund schemes.

**Deliverables produced:**
- `reports/fund_scorecard.csv`
- `reports/alpha_beta.csv`
- `reports/benchmark_comparison.png`

In [ ]:
"""
Bluestock Fintech — Performance Analytics
"""

import os
import sys
import warnings
from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg") 
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")


sys.stdout.reconfigure(encoding="utf-8", errors="replace")

PROJECT_ROOT = Path(__file__).resolve().parent.parent
DATA_DIR     = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR  = PROJECT_ROOT / "reports"
CHARTS_DIR   = REPORTS_DIR / "charts"



print("=" * 72)
print("  BLUESTOCK FINTECH — PERFORMANCE ANALYTICS")
print("=" * 72)



## SECTION 1 — DATA LOADING & PREPARATION

In [ ]:
print("\n[1/9] Loading data …")

fund_master = pd.read_csv(DATA_DIR / "01_fund_master.csv")
nav_history = pd.read_csv(DATA_DIR / "02_nav_history.csv", parse_dates=["date"])
benchmarks  = pd.read_csv(DATA_DIR / "10_benchmark_indices.csv", parse_dates=["date"])

print(f"     Fund Master : {len(fund_master)} schemes")
print(f"     NAV History : {len(nav_history):,} rows  |  "
      f"{nav_history['amfi_code'].nunique()} funds  |  "
      f"{nav_history['date'].min().date()} → {nav_history['date'].max().date()}")
print(f"     Benchmarks  : {benchmarks['index_name'].nunique()} indices  |  "
      f"{benchmarks['date'].min().date()} → {benchmarks['date'].max().date()}")

# Pivot NAV history: rows = dates, columns = amfi_code
nav_wide = nav_history.pivot(index="date", columns="amfi_code", values="nav").sort_index()
nav_wide = nav_wide.dropna(how="all")

# Pivot benchmarks: rows = dates, columns = index_name
bm_wide = benchmarks.pivot(index="date", columns="index_name", values="close_value").sort_index()

# Build a mapping: amfi_code → benchmark index name in our data
# Map the fund_master benchmark names to the column names in benchmarks CSV
BENCHMARK_MAP = {
    "NIFTY 100 TRI":              "NIFTY100",
    "NIFTY 50 TRI":               "NIFTY50",
    "NIFTY 500 TRI":              "NIFTY500",
    "NIFTY Midcap 150 TRI":       "NIFTY_MIDCAP150",
    "NIFTY Midcap 50 TRI":        "NIFTY_MIDCAP150",  # map to closest available
    "BSE 250 SmallCap TRI":       "BSE_SMALLCAP",
    "CRISIL Dynamic Gilt Index":  "CRISIL_GILT",
    "CRISIL Liquid Fund AI Index":"CRISIL_LIQUID",
    "CRISIL Short Term Bond Index":"CRISIL_LIQUID",  # closest proxy
    "NIFTY Large Midcap 250 TRI": "NIFTY500",         # closest proxy
}

fund_master["bm_col"] = fund_master["benchmark"].map(BENCHMARK_MAP)

# Create lookup dicts
code_to_name   = dict(zip(fund_master["amfi_code"], fund_master["scheme_name"]))
code_to_cat    = dict(zip(fund_master["amfi_code"], fund_master["sub_category"]))
code_to_plan   = dict(zip(fund_master["amfi_code"], fund_master["plan"]))
code_to_er     = dict(zip(fund_master["amfi_code"], fund_master["expense_ratio_pct"]))
code_to_bm     = dict(zip(fund_master["amfi_code"], fund_master["bm_col"]))
code_to_fh     = dict(zip(fund_master["amfi_code"], fund_master["fund_house"]))

ALL_CODES = sorted(fund_master["amfi_code"].tolist())
LATEST_DATE = nav_wide.index.max()

print(f"     Latest NAV date : {LATEST_DATE.date()}")
print(f"     NAV matrix shape: {nav_wide.shape}")



## SECTION 2 — DAILY RETURNS

In [ ]:
print("\n[2/9] Computing daily returns …")

# Fund daily returns
fund_returns = nav_wide.pct_change().dropna(how="all")

# Benchmark daily returns
bm_returns = bm_wide.pct_change().dropna(how="all")

# Validation: summary statistics
ret_stats = fund_returns.describe().T
ret_stats.index.name = "amfi_code"
ret_stats["skew"]     = fund_returns.skew()
ret_stats["kurtosis"] = fund_returns.kurtosis()

print("     Daily return summary (across all funds):")
print(f"       Mean of means : {ret_stats['mean'].mean():.6f}")
print(f"       Mean of stds  : {ret_stats['std'].mean():.6f}")
print(f"       Mean skewness : {ret_stats['skew'].mean():.4f}")
print(f"       Mean kurtosis : {ret_stats['kurtosis'].mean():.4f}")

# Check for extreme outliers (|return| > 15%)
outlier_count = (fund_returns.abs() > 0.15).sum().sum()
print(f"       Outliers (|r| > 15%) : {outlier_count}")
print("     ✓ Daily returns look reasonable." if outlier_count < 10
      else f"     ⚠ {outlier_count} extreme returns detected.")



## SECTION 3 — CAGR (1yr, 3yr, 5yr)

In [ ]:
print("\n[3/9] Computing CAGR (1yr, 3yr, 5yr) …")

def compute_cagr(nav_series, start_date, end_date):
    """Compute CAGR between two dates, finding nearest available trading days."""
    available = nav_series.dropna()
    if len(available) == 0:
        return np.nan

    # Find nearest trading day on or after start_date
    valid_start = available.index[available.index >= start_date]
    if len(valid_start) == 0:
        return np.nan
    actual_start = valid_start[0]

    # Find nearest trading day on or before end_date
    valid_end = available.index[available.index <= end_date]
    if len(valid_end) == 0:
        return np.nan
    actual_end = valid_end[-1]

    nav_start = available[actual_start]
    nav_end   = available[actual_end]

    if nav_start <= 0 or pd.isna(nav_start) or pd.isna(nav_end):
        return np.nan

    n_years = (actual_end - actual_start).days / 365.25
    if n_years < 0.5:  
        return np.nan

    cagr = (nav_end / nav_start) ** (1 / n_years) - 1
    return cagr

# Lookback dates
end_date = LATEST_DATE
start_1yr = end_date - timedelta(days=365)
start_3yr = end_date - timedelta(days=3 * 365)
start_5yr = end_date - timedelta(days=5 * 365)

cagr_data = []
for code in ALL_CODES:
    if code not in nav_wide.columns:
        continue
    nav_s = nav_wide[code]
    cagr_1yr = compute_cagr(nav_s, start_1yr, end_date)
    cagr_3yr = compute_cagr(nav_s, start_3yr, end_date)
    cagr_5yr = compute_cagr(nav_s, start_5yr, end_date)
    cagr_data.append({
        "amfi_code":     code,
        "scheme_name":   code_to_name.get(code, ""),
        "fund_house":    code_to_fh.get(code, ""),
        "category":      code_to_cat.get(code, ""),
        "plan":          code_to_plan.get(code, ""),
        "cagr_1yr_pct":  round(cagr_1yr * 100, 2) if not np.isnan(cagr_1yr) else np.nan,
        "cagr_3yr_pct":  round(cagr_3yr * 100, 2) if not np.isnan(cagr_3yr) else np.nan,
        "cagr_5yr_pct":  round(cagr_5yr * 100, 2) if not np.isnan(cagr_5yr) else np.nan,
    })

cagr_df = pd.DataFrame(cagr_data)

print("     CAGR Comparison Table (top 10 by 3yr CAGR):")
top10 = cagr_df.nlargest(10, "cagr_3yr_pct")[
    ["scheme_name", "category", "cagr_1yr_pct", "cagr_3yr_pct", "cagr_5yr_pct"]
]
print(top10.to_string(index=False))



## SECTION 4 — SHARPE RATIO

In [ ]:
print("\n[4/9] Computing Sharpe Ratios …")

RF_ANNUAL = 0.065  
RF_DAILY  = RF_ANNUAL / 252

sharpe_data = []
for code in ALL_CODES:
    if code not in fund_returns.columns:
        continue
    rets = fund_returns[code].dropna()
    if len(rets) < 60:  
        sharpe_data.append({"amfi_code": code, "sharpe_ratio": np.nan})
        continue
    excess = rets - RF_DAILY
    sharpe = (excess.mean() / excess.std()) * np.sqrt(252)
    sharpe_data.append({"amfi_code": code, "sharpe_ratio": round(sharpe, 4)})

sharpe_df = pd.DataFrame(sharpe_data)
sharpe_df = sharpe_df.sort_values("sharpe_ratio", ascending=False).reset_index(drop=True)
sharpe_df["sharpe_rank"] = range(1, len(sharpe_df) + 1)

print("     Top 10 by Sharpe Ratio:")
for _, row in sharpe_df.head(10).iterrows():
    name = code_to_name.get(row["amfi_code"], "")[:50]
    print(f"       #{int(row['sharpe_rank']):2d}  Sharpe={row['sharpe_ratio']:7.4f}  {name}")



## SECTION 5 — SORTINO RATIO

In [ ]:
print("\n[5/9] Computing Sortino Ratios …")

sortino_data = []
for code in ALL_CODES:
    if code not in fund_returns.columns:
        continue
    rets = fund_returns[code].dropna()
    if len(rets) < 60:
        sortino_data.append({"amfi_code": code, "sortino_ratio": np.nan})
        continue
    excess = rets - RF_DAILY
    downside = rets[rets < 0]
    if len(downside) < 10:
        sortino_data.append({"amfi_code": code, "sortino_ratio": np.nan})
        continue
    downside_std = downside.std()
    sortino = (excess.mean() / downside_std) * np.sqrt(252)
    sortino_data.append({"amfi_code": code, "sortino_ratio": round(sortino, 4)})

sortino_df = pd.DataFrame(sortino_data)
sortino_df = sortino_df.sort_values("sortino_ratio", ascending=False).reset_index(drop=True)
sortino_df["sortino_rank"] = range(1, len(sortino_df) + 1)

print("     Top 10 by Sortino Ratio:")
for _, row in sortino_df.head(10).iterrows():
    name = code_to_name.get(row["amfi_code"], "")[:50]
    print(f"       #{int(row['sortino_rank']):2d}  Sortino={row['sortino_ratio']:7.4f}  {name}")



## SECTION 6 — ALPHA & BETA (OLS REGRESSION)

In [ ]:
print("\n[6/9] Computing Alpha & Beta via OLS regression …")

alpha_beta_data = []
for code in ALL_CODES:
    if code not in fund_returns.columns:
        continue

    bm_col = code_to_bm.get(code, "NIFTY100")
    if bm_col not in bm_returns.columns:
        bm_col = "NIFTY100"  # fallback

    # Align fund and benchmark returns on common dates
    common_idx = fund_returns[code].dropna().index.intersection(
        bm_returns[bm_col].dropna().index
    )
    if len(common_idx) < 60:
        alpha_beta_data.append({
            "amfi_code": code, "alpha_ann": np.nan, "beta": np.nan,
            "r_squared": np.nan, "benchmark_used": bm_col
        })
        continue

    fund_r = fund_returns[code].loc[common_idx].values
    bm_r   = bm_returns[bm_col].loc[common_idx].values

    slope, intercept, r_value, p_value, std_err = stats.linregress(bm_r, fund_r)

    alpha_ann = intercept * 252 
    beta      = slope
    r_sq      = r_value ** 2

    alpha_beta_data.append({
        "amfi_code":      code,
        "scheme_name":    code_to_name.get(code, ""),
        "fund_house":     code_to_fh.get(code, ""),
        "category":       code_to_cat.get(code, ""),
        "plan":           code_to_plan.get(code, ""),
        "alpha_ann":      round(alpha_ann, 4),
        "beta":           round(beta, 4),
        "r_squared":      round(r_sq, 4),
        "p_value":        round(p_value, 6),
        "benchmark_used": bm_col,
    })

alpha_beta_df = pd.DataFrame(alpha_beta_data)

# Save alpha_beta.csv
ab_save = alpha_beta_df.copy()
ab_save_path = REPORTS_DIR / "alpha_beta.csv"
ab_save.to_csv(ab_save_path, index=False)
print(f"     ✓ Saved: {ab_save_path}")

print("     Top 10 by Alpha (annualized):")
top_alpha = alpha_beta_df.nlargest(10, "alpha_ann")
for _, row in top_alpha.iterrows():
    name = code_to_name.get(row["amfi_code"], "")[:45]
    print(f"       α={row['alpha_ann']:+7.4f}  β={row['beta']:.4f}  R²={row['r_squared']:.4f}  {name}")



## SECTION 7 — MAXIMUM DRAWDOWN

In [ ]:
print("\n[7/9] Computing Maximum Drawdowns …")

dd_data = []
for code in ALL_CODES:
    if code not in nav_wide.columns:
        continue
    nav_s = nav_wide[code].dropna()
    if len(nav_s) < 20:
        dd_data.append({
            "amfi_code": code, "max_drawdown_pct": np.nan,
            "dd_peak_date": None, "dd_trough_date": None
        })
        continue

    running_max = nav_s.cummax()
    drawdown    = nav_s / running_max - 1

    max_dd      = drawdown.min()
    trough_date = drawdown.idxmin()

    # Peak date = last date before trough where NAV was at running max
    peak_region = running_max.loc[:trough_date]
    peak_date   = peak_region.index[peak_region == peak_region.iloc[-1]][0]

    dd_data.append({
        "amfi_code":       code,
        "max_drawdown_pct": round(max_dd * 100, 2),
        "dd_peak_date":    peak_date.date(),
        "dd_trough_date":  trough_date.date(),
    })

dd_df = pd.DataFrame(dd_data)

print("     Worst 10 drawdowns:")
worst10 = dd_df.nsmallest(10, "max_drawdown_pct")
for _, row in worst10.iterrows():
    name = code_to_name.get(row["amfi_code"], "")[:45]
    print(f"       DD={row['max_drawdown_pct']:+7.2f}%  "
          f"({row['dd_peak_date']} → {row['dd_trough_date']})  {name}")



## SECTION 8 — FUND SCORECARD (0–100)

In [ ]:
print("\n[8/9] Building Fund Scorecard …")

# Merge all metrics into one DataFrame
scorecard = fund_master[["amfi_code", "scheme_name", "fund_house", "category",
                         "sub_category", "plan", "expense_ratio_pct"]].copy()

# Merge CAGR
scorecard = scorecard.merge(
    cagr_df[["amfi_code", "cagr_1yr_pct", "cagr_3yr_pct", "cagr_5yr_pct"]],
    on="amfi_code", how="left"
)

# Merge Sharpe
scorecard = scorecard.merge(
    sharpe_df[["amfi_code", "sharpe_ratio"]],
    on="amfi_code", how="left"
)

# Merge Sortino
scorecard = scorecard.merge(
    sortino_df[["amfi_code", "sortino_ratio"]],
    on="amfi_code", how="left"
)

# Merge Alpha & Beta
scorecard = scorecard.merge(
    alpha_beta_df[["amfi_code", "alpha_ann", "beta", "r_squared"]],
    on="amfi_code", how="left"
)

# Merge Max Drawdown
scorecard = scorecard.merge(
    dd_df[["amfi_code", "max_drawdown_pct", "dd_peak_date", "dd_trough_date"]],
    on="amfi_code", how="left"
)


n = len(scorecard)


def percentile_rank(series, ascending=True):
    """Rank values as percentile 0–100. ascending=True means higher value → higher rank."""
    ranked = series.rank(ascending=ascending, na_option="bottom", pct=True)
    return ranked * 100


scorecard["rank_3yr_return"]   = percentile_rank(scorecard["cagr_3yr_pct"], ascending=True)
scorecard["rank_sharpe"]       = percentile_rank(scorecard["sharpe_ratio"], ascending=True)
scorecard["rank_alpha"]        = percentile_rank(scorecard["alpha_ann"],    ascending=True)
scorecard["rank_expense_inv"]  = percentile_rank(scorecard["expense_ratio_pct"], ascending=False)
scorecard["rank_dd_inv"]       = percentile_rank(scorecard["max_drawdown_pct"], ascending=True)


# Composite score
scorecard["score"] = (
    0.30 * scorecard["rank_3yr_return"]
  + 0.25 * scorecard["rank_sharpe"]
  + 0.20 * scorecard["rank_alpha"]
  + 0.15 * scorecard["rank_expense_inv"]
  + 0.10 * scorecard["rank_dd_inv"]
).round(2)

scorecard = scorecard.sort_values("score", ascending=False).reset_index(drop=True)
scorecard["overall_rank"] = range(1, len(scorecard) + 1)

# Save fund_scorecard.csv
sc_save_path = REPORTS_DIR / "fund_scorecard.csv"
scorecard.to_csv(sc_save_path, index=False)
print(f"     ✓ Saved: {sc_save_path}")

print("\n     Fund Scorecard — All 40 Funds:")
print("     " + "-" * 90)
print(f"     {'Rank':>4s}  {'Score':>5s}  {'Sharpe':>6s}  {'3yr%':>6s}  {'Alpha':>7s}  "
      f"{'MaxDD%':>7s}  {'ER%':>5s}  Scheme Name")
print("     " + "-" * 90)
for _, row in scorecard.iterrows():
    name = row["scheme_name"][:48]
    print(f"     {row['overall_rank']:4d}  {row['score']:5.1f}  "
          f"{row['sharpe_ratio']:6.3f}  {row['cagr_3yr_pct']:6.2f}  "
          f"{row['alpha_ann']:+7.4f}  {row['max_drawdown_pct']:+7.2f}  "
          f"{row['expense_ratio_pct']:5.2f}  {name}")



## SECTION 9 — BENCHMARK COMPARISON CHART

In [ ]:
print("\n[9/9] Generating Benchmark Comparison Chart …")

# Top 5 funds by scorecard
top5_codes = scorecard.head(5)["amfi_code"].tolist()
top5_names = [code_to_name[c] for c in top5_codes]

# 3-year window
chart_start = end_date - timedelta(days=3 * 365)

# Fund NAV series (normalized to 100)
fig, ax = plt.subplots(figsize=(14, 8))

# Color palette — premium look
colors = ["#6366F1", "#F59E0B", "#10B981", "#EF4444", "#8B5CF6",
          "#374151", "#0EA5E9"]

# Plot top 5 funds
tracking_errors = {}
for i, code in enumerate(top5_codes):
    nav_s = nav_wide[code].dropna()
    nav_s = nav_s[nav_s.index >= chart_start]
    if len(nav_s) < 20:
        continue
    normalized = nav_s / nav_s.iloc[0] * 100
    short_name = code_to_name[code].replace(" - Regular Plan", "").replace(" - Regular", "")
    short_name = short_name.replace(" - Growth", "").replace(" Fund", "")
    ax.plot(normalized.index, normalized.values, label=short_name,
            color=colors[i], linewidth=2, alpha=0.9)

    # Compute tracking error vs NIFTY100
    bm_col = "NIFTY100"
    common = fund_returns[code].dropna().index.intersection(bm_returns[bm_col].dropna().index)
    common = common[common >= chart_start]
    if len(common) > 60:
        diff = fund_returns[code].loc[common] - bm_returns[bm_col].loc[common]
        te = diff.std() * np.sqrt(252)
        tracking_errors[code_to_name[code]] = round(te * 100, 2)

# Plot NIFTY 50
nifty50_s = bm_wide["NIFTY50"].dropna()
nifty50_s = nifty50_s[nifty50_s.index >= chart_start]
if len(nifty50_s) > 0:
    n50_norm = nifty50_s / nifty50_s.iloc[0] * 100
    ax.plot(n50_norm.index, n50_norm.values, label="NIFTY 50",
            color=colors[5], linewidth=2.5, linestyle="--", alpha=0.8)

# Plot NIFTY 100
nifty100_s = bm_wide["NIFTY100"].dropna()
nifty100_s = nifty100_s[nifty100_s.index >= chart_start]
if len(nifty100_s) > 0:
    n100_norm = nifty100_s / nifty100_s.iloc[0] * 100
    ax.plot(n100_norm.index, n100_norm.values, label="NIFTY 100",
            color=colors[6], linewidth=2.5, linestyle="--", alpha=0.8)

# Styling
ax.set_title("Top 5 Funds vs Benchmarks — 3-Year Cumulative Returns",
             fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("Date", fontsize=12)
ax.set_ylabel("Normalized Value (Base = 100)", fontsize=12)
ax.legend(loc="upper left", fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle="--")
ax.set_facecolor("#FAFAFA")
fig.patch.set_facecolor("#FFFFFF")

# Format y-axis as integers
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f"))

plt.tight_layout()
chart_path = CHARTS_DIR / "benchmark_comparison.png"
fig.savefig(chart_path, dpi=150, bbox_inches="tight", facecolor="#FFFFFF")
plt.close(fig)
print(f"     ✓ Saved: {chart_path}")

print("\n     Tracking Error vs NIFTY 100 (annualized, 3yr window):")
for name, te in tracking_errors.items():
    short = name[:55]
    print(f"       TE = {te:6.2f}%  {short}")



## SUMMARY

In [ ]:
print("\n" + "=" * 72)
print("  DELIVERABLES GENERATED")
print("=" * 72)
print(f"  1. fund_scorecard.csv    → {sc_save_path}")
print(f"  2. alpha_beta.csv        → {ab_save_path}")
print(f"  3. benchmark_comparison  → {chart_path}")
print("=" * 72)
print("  Done!")
